# 02 — Embedding Extraction

Frozen **V-JEPA2** → one L2-normalized vector per InHARD clip.

Output: `outputs/embeddings.npz`, `outputs/embedding_meta.csv`

In [ ]:
import sys
from pathlib import Path
NB = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
sys.path.insert(0, str(NB))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from lib.embeddings import extract_clip_from_file, extract_vjepa_embedding, get_device
from lib.inhard import analyze_training_clips
from lib.paths import OUTPUTS_DIR, find_inhard_root

MAX_CLIPS = 50
print("Device:", get_device())

In [ ]:
report = analyze_training_clips(find_inhard_root(), min_classes=1, max_clips=MAX_CLIPS)
if not report.ok:
    raise FileNotFoundError(report.error)
clips = report.clips
classes = sorted({c.label for c in clips})
label_to_idx = {c: i for i, c in enumerate(classes)}
print(len(clips), "clips,", len(classes), "classes")

In [ ]:
X_list, y_list, meta = [], [], []
for rec in tqdm(clips):
    frames = extract_clip_from_file(rec.path)
    if len(frames) < 4:
        continue
    emb = extract_vjepa_embedding(frames)
    X_list.append(emb)
    y_list.append(label_to_idx[rec.label])
    meta.append({"path": str(rec.path), "label": rec.label})

X = np.stack(X_list).astype(np.float32)
y = np.array(y_list, dtype=np.int64)
np.savez(OUTPUTS_DIR / "embeddings.npz", X=X, y=y, class_names=np.array(classes, dtype=object))
pd.DataFrame(meta).to_csv(OUTPUTS_DIR / "embedding_meta.csv", index=False)
print("Saved", X.shape)